1. Read fact tables

In [0]:
df_fato_arboviroses = spark.table("fato_arboviroses")
df_fato_covid = spark.table("fato_covid")
df_fato_escorpiao = spark.table("fato_escorpiao")
df_fato_dengue_espacial = spark.table("fato_dengue_espacial")
df_fato_chikungunya_espacial = spark.table("fato_chikungunya_espacial")

2. Creates dim_tempo

In [0]:
from pyspark.sql.functions import col, concat_ws, lpad

df_dim_tempo = (
    df_fato_arboviroses
    .select("ano", "mes", "ordem_mes")
    .unionByName(
        df_fato_covid.select("ano", "mes", "ordem_mes")
    )
    .unionByName(
        df_fato_escorpiao.select("ano", "mes", "ordem_mes")
    )
    .dropDuplicates()
    .withColumn(
        "id_tempo",
        concat_ws(
            "",
            col("ano").cast("string"),
            lpad(col("ordem_mes").cast("string"), 2, "0")
        ).cast("int")
    )
    .withColumn(
        "ano_mes",
        concat_ws(
            "-",
            col("ano").cast("string"),
            lpad(col("ordem_mes").cast("string"), 2, "0")
        )
    )
    .select("id_tempo", "ano", "mes", "ordem_mes", "ano_mes")
    .orderBy("ano", "ordem_mes")
)

df_dim_tempo.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_tempo")

display(
    spark.table("dim_tempo")
    .orderBy("ano", "ordem_mes")
)

3. Creates dim_doenca

In [0]:
from pyspark.sql.functions import col, lower, trim
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_doencas_base = (
    df_fato_arboviroses
    .select("doenca")
    .unionByName(df_fato_covid.select("doenca"))
    .unionByName(df_fato_dengue_espacial.select("doenca"))
    .unionByName(df_fato_chikungunya_espacial.select("doenca"))
    .dropDuplicates()
    .withColumn("doenca", lower(trim(col("doenca"))))
)

df_escorpiao_manual = spark.createDataFrame(
    [("escorpiao",)],
    ["doenca"]
)

df_dim_doenca = (
    df_doencas_base
    .unionByName(df_escorpiao_manual)
    .dropDuplicates()
)

window_doenca = Window.orderBy("doenca")

df_dim_doenca = (
    df_dim_doenca
    .withColumn("id_doenca", row_number().over(window_doenca))
    .select("id_doenca", "doenca")
    .orderBy("id_doenca")
)

df_dim_doenca.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_doenca")

display(
    spark.table("dim_doenca")
    .orderBy("id_doenca")
)

4. Creates dim_bairro

In [0]:
from pyspark.sql.functions import col, lower, trim, row_number
from pyspark.sql.window import Window

df_dim_bairro = (
    df_fato_dengue_espacial
    .select("bairro")
    .unionByName(
        df_fato_chikungunya_espacial.select("bairro")
    )
    .filter(col("bairro").isNotNull())
    .withColumn("bairro", lower(trim(col("bairro"))))
    .filter(col("bairro") != "")
    .dropDuplicates()
)

window_bairro = Window.orderBy("bairro")

df_dim_bairro = (
    df_dim_bairro
    .withColumn("id_bairro", row_number().over(window_bairro))
    .select("id_bairro", "bairro")
    .orderBy("id_bairro")
)

display(df_dim_bairro)

df_dim_bairro.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_bairro")

5. Creates dim_sexo

In [0]:
from pyspark.sql.functions import col, trim, upper, row_number
from pyspark.sql.window import Window

df_dim_sexo = (
    df_fato_dengue_espacial
    .select("sexo")
    .unionByName(df_fato_chikungunya_espacial.select("sexo"))
    .filter(col("sexo").isNotNull())
    .withColumn("sexo", trim(upper(col("sexo"))))
    .filter(col("sexo") != "")
    .dropDuplicates()
)

window_sexo = Window.orderBy("sexo")

df_dim_sexo = (
    df_dim_sexo
    .withColumn("id_sexo", row_number().over(window_sexo))
    .select("id_sexo", "sexo")
    .orderBy("id_sexo")
)

display(df_dim_sexo)

df_dim_sexo.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_sexo")

6. Load dimensões

In [0]:
df_dim_tempo = spark.table("dim_tempo")
df_dim_doenca = spark.table("dim_doenca")
df_dim_bairro = spark.table("dim_bairro")
df_dim_sexo = spark.table("dim_sexo")

7. Model Arboviroses

In [0]:
df_fato_arboviroses_final = (
    df_fato_arboviroses
    .join(df_dim_tempo, on=["ano", "mes", "ordem_mes"], how="left")
    .join(df_dim_doenca, on="doenca", how="left")
    .select(
        "id_tempo",
        "id_doenca",
        "casos",
        "obitos"
    )
)

display(df_fato_arboviroses_final)

df_fato_arboviroses_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fato_arboviroses_final")

8. Model Covid

In [0]:
df_fato_covid_final = (
    df_fato_covid
    .join(df_dim_tempo, on=["ano", "mes", "ordem_mes"], how="left")
    .join(df_dim_doenca, on="doenca", how="left")
    .select(
        "id_tempo",
        "id_doenca",
        "casos",
        "obitos"
    )
)

display(df_fato_covid_final)

df_fato_covid_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fato_covid_final")

9. Model Escorpiao

In [0]:
from pyspark.sql.functions import lit

df_fato_escorpiao_final = (
    df_fato_escorpiao
    .withColumn("doenca", lit("escorpiao"))
    .join(df_dim_tempo, on=["ano", "mes", "ordem_mes"], how="left")
    .join(df_dim_doenca, on="doenca", how="left")
    .select(
        "id_tempo",
        "id_doenca",
        "casos",
        "obitos"
    )
)

display(df_fato_escorpiao_final)

df_fato_escorpiao_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fato_escorpiao_final")

10. Model dengue espacial

In [0]:
from pyspark.sql.functions import col, lower, trim

df_fato_dengue_espacial_final = (
    df_fato_dengue_espacial
    .withColumn("doenca", lower(trim(col("doenca"))))
    .withColumn("bairro", lower(trim(col("bairro"))))
    .join(df_dim_doenca, on="doenca", how="left")
    .join(df_dim_bairro, on="bairro", how="left")
    .join(df_dim_sexo, on="sexo", how="left")
    .select(
        "id_doenca",
        "id_bairro",
        "id_sexo",
        "data",
        "idade",
        "exame"
    )
)

display(df_fato_dengue_espacial_final)

df_fato_dengue_espacial_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fato_dengue_espacial_final")

11. Model chikungunya espacial

In [0]:
from pyspark.sql.functions import col, lower, trim

df_fato_chikungunya_espacial_final = (
    df_fato_chikungunya_espacial
    .withColumn("doenca", lower(trim(col("doenca"))))
    .withColumn("bairro", lower(trim(col("bairro"))))
    .join(df_dim_doenca, on="doenca", how="left")
    .join(df_dim_bairro, on="bairro", how="left")
    .join(df_dim_sexo, on="sexo", how="left")
    .select(
        "id_doenca",
        "id_bairro",
        "id_sexo",
        "idade"
    )
)

display(df_fato_chikungunya_espacial_final)

df_fato_chikungunya_espacial_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fato_chikungunya_espacial_final")

12. Final check

In [0]:
print("=== VALIDACAO DAS FATOS FINAIS ===")

# 1) FATO ARBOVIROSES FINAL
print("\n--- fato_arboviroses_final ---")
df_fato_arboviroses_final.filter(
    "id_tempo IS NULL OR id_doenca IS NULL"
).show()

print(
    "contagem original x final:",
    df_fato_arboviroses.count(),
    df_fato_arboviroses_final.count()
)

# 2) FATO COVID FINAL
print("\n--- fato_covid_final ---")
df_fato_covid_final.filter(
    "id_tempo IS NULL OR id_doenca IS NULL"
).show()

print(
    "contagem original x final:",
    df_fato_covid.count(),
    df_fato_covid_final.count()
)

# 3) FATO ESCORPIAO FINAL
print("\n--- fato_escorpiao_final ---")
df_fato_escorpiao_final.filter(
    "id_tempo IS NULL OR id_doenca IS NULL"
).show()

print(
    "contagem original x final:",
    df_fato_escorpiao.count(),
    df_fato_escorpiao_final.count()
)

# 4) FATO DENGUE ESPACIAL FINAL
print("\n--- fato_dengue_espacial_final ---")
df_fato_dengue_espacial_final.filter(
    "id_doenca IS NULL OR id_bairro IS NULL OR id_sexo IS NULL"
).show()

print(
    "contagem original x final:",
    df_fato_dengue_espacial.count(),
    df_fato_dengue_espacial_final.count()
)

# 5) FATO CHIKUNGUNYA ESPACIAL FINAL
print("\n--- fato_chikungunya_espacial_final ---")
df_fato_chikungunya_espacial_final.filter(
    "id_doenca IS NULL OR id_bairro IS NULL OR id_sexo IS NULL"
).show()

print(
    "contagem original x final:",
    df_fato_chikungunya_espacial.count(),
    df_fato_chikungunya_espacial_final.count()
)

Another check

In [0]:
# DIAGNOSTICO DE BAIRROS SEM MATCH

from pyspark.sql.functions import col, lower, trim

print("=== BAIRROS SEM MATCH NO DENGUE ESPACIAL ===")
display(
    df_fato_dengue_espacial
    .withColumn("bairro", lower(trim(col("bairro"))))
    .join(df_dim_bairro, on="bairro", how="left")
    .filter(col("id_bairro").isNull())
    .select("bairro")
    .distinct()
    .orderBy("bairro")
)

print("=== BAIRROS SEM MATCH NO CHIKUNGUNYA ESPACIAL ===")
display(
    df_fato_chikungunya_espacial
    .withColumn("bairro", lower(trim(col("bairro"))))
    .join(df_dim_bairro, on="bairro", how="left")
    .filter(col("id_bairro").isNull())
    .select("bairro")
    .distinct()
    .orderBy("bairro")
)